<a href="https://colab.research.google.com/github/tuankhoin/CO3133-Deep-Learning/blob/main/Week_2_Foundation_Knowledges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3133 - Deep Learning and Its Applications

# Week 2 - Foundation Knowledges

### From Problem Formulation to Training Neural Networks

This lecture develops the foundations required for later deep-learning topics:

- formulate a machine-learning problem;
- represent and split datasets correctly;
- understand regression and classification models;
- distinguish losses from evaluation metrics;
- reason about model capacity and generalization;
- understand multilayer perceptrons;
- understand how neural networks are trained.

## Lecture Contents

1. Problem Formulation
2. Datasets
3. Regression Summary
4. Classification Summary
5. Loss Functions
6. Evaluation Metrics
7. Model Capacity and Inductive Bias
8. Multilayer Perceptron (MLP)
9. Training Neural Networks

---
# 1. Problem Formulation
---

## From Data to a Learning Problem

Given a dataset $
\mathcal{D}=\{(\mathbf{x}_i,y_i)\}_{i=1}^{N},
$

we want to learn a parameterized function $
f_\theta(\mathbf{x}) \approx y.
$

Machine learning is about **learning a mapping from data**, rather than manually writing all rules.

<img src="https://i.programmerhumor.io/2025/03/f21ce9e6749a58c6ab25fb434fc93f5428d079e5088466487c0026486df029dd.jpeg" height=300/>

## Notation Convention

| Symbol | Meaning |
|---|---|
| $\mathbf{x}$ | Single input vector |
| $\mathbf{X}$ | Batch / matrix / tensor of inputs |
| $y,\mathbf{y}$ | Ground-truth target |
| $\hat y,\hat{\mathbf{Y}}$ | Prediction |
| $\mathbf{Z}$ | Logits |
| $\hat{\mathbf{P}}$ | Predicted probabilities |
| $\theta$ | All learnable parameters |
| $\mathbf{W},\mathbf{b}$ | Weights and biases |
| $B$ | Batch size |
| $D$ | Input dimension |
| $K$ | Number of outputs/classes |
| $N$ | Total samples |

Samples are stacked as rows:

$$
\mathbf{X}\in\mathbb{R}^{B\times D}.
$$

Typical deep-learning shapes:

- image: $(B,C,H,W)$
- sequence: $(B,T)$
- embedded sequence: $(B,T,D)$

## Supervised Learning and Output Spaces

| Task | Output space --- |
|---|---|
| Scalar regression | $y\in\mathbb{R}$ |
| Vector regression | $\mathbf{y}\in\mathbb{R}^K$ |
| Binary classification | $y\in\{0,1\}$ |
| Single-label multiclass | $y\in\{1,\ldots,K\}$ |
| Multi-label classification | $\mathbf{y}\in\{0,1\}^K$ |

The **output space determines the output activation and loss function**.

## Learning Objective: Minimize the **Loss Function**

<img src="https://i.redd.it/the-loss-function-v0-f5hw1cwi9mlb1.png?width=640&format=png&auto=webp&s=1c8a5e9cf6204b5ae05c07865f5e7662037ba539" height=300/>

Learn parameters by minimizing a loss:

$$
\theta^*=\arg\min_\theta \mathcal{L}(f_\theta(\mathbf{X}),\mathbf{Y}).
$$

Assume data come from an unknown distribution:

$$
(\mathbf{x},y)\sim P(\mathbf{x},y).
$$

The ideal objective is the expected risk:

$$
\mathcal{R}(\theta)=\mathbb{E}_{(\mathbf{x},y)\sim P}[\ell(f_\theta(\mathbf{x}),y)].
$$

What we actually minimize is empirical risk:

$$
\hat{\mathcal{R}}(\theta)=\frac{1}{N}\sum_{i=1}^{N}\ell(f_\theta(\mathbf{x}_i),y_i).
$$

**Low training error does not guarantee good generalization.**

## Training, Validation, Test and Inductive Bias

- **Training set:** learn parameters.
- **Validation set:** tune hyperparameters, model selection, early stopping.
- **Test set:** final unbiased evaluation.

> The test set must not influence training or tuning.

A model is written as $
f_\theta(\mathbf{x})
$

**Inductive bias** = assumptions built into the model.

Examples:

- Linear model → linear relationships in feature space
- CNN → locality and spatial structure
- Transformer → interactions through attention

Learning requires some form of inductive bias.

---
# 2. Preparing Datasets
---

A dataset is more than a collection of files. Its representation, quality, splitting strategy and loading pipeline all affect learning.

## Samples and Common Data Shapes

A feature vector:

$$
\mathbf{x}=\begin{bmatrix}x_1\\x_2\\\vdots\\x_d\end{bmatrix}\in\mathbb{R}^{d}.
$$

| Data | Single sample | Batch --------|
|---|---|---|
| Tabular | $\mathbb{R}^D$ | $\mathbb{R}^{B\times D}$ |
| Image | $\mathbb{R}^{C\times H\times W}$ | $\mathbb{R}^{B\times C\times H\times W}$ |
| Token sequence | $\mathbb{R}^{T}$ | $\mathbb{R}^{B\times T_{\max}}$ |
| Embedded sequence | $\mathbb{R}^{T\times D}$ | $\mathbb{R}^{B\times T_{\max}\times D}$ |
| Time series | $\mathbb{R}^{T\times F}$ | $\mathbb{R}^{B\times T\times F}$ |

Variable-length sequences are usually **padded** and accompanied by a **mask**.

## Data Quality and Preprocessing

![](https://media.geeksforgeeks.org/wp-content/uploads/20250527114301078238/Missing-Values.png)

Common problems:

- missing values;
- noise and outliers;
- incorrect labels;
- duplicate samples;
- biased or unrepresentative data.

Standardization:

$$
\tilde{x}_j=\frac{x_j-\mu_j}{\sigma_j}.
$$

Important rule:

> Compute $\mu$, $\sigma$, encoders and other preprocessing statistics using the **training set only**.

Otherwise information leaks from validation/test data.

## Dataset Splitting

<img src="https://miro.medium.com/0*PrCHWNCUTo-z_LSv.jpg" height=300/>

Typical examples:

- 70% train / 15% validation / 15% test
- 80% train / 10% validation / 10% test

### Classification

Use **stratification** where possible so each split has similar class proportions.

### Limited data

Use $K$-fold cross-validation for model selection. The test set remains held out.

### Temporal data

Do not randomly mix future samples into training:

$$
\text{past}\rightarrow\text{validation}\rightarrow\text{future test}.
$$

## Mini-batches, Shuffling and Data Leakage

Instead of using all $N$ samples at once, use only part of data called batch: $
\mathbf{X}_b\in\mathbb{R}^{B\times D}
$

Mini-batches:

- reduce memory usage;
- improve GPU utilization;
- provide noisy gradient estimates;
- make SGD-style training practical.

For ordinary i.i.d. datasets, samples are normally shuffled every epoch.

Examples of **leakage**:

- normalizing using the complete dataset;
- performing feature selection using test labels;
- putting near-duplicate samples in train and test;
- generating augmentations before splitting incorrectly.

> **Test data must remain untouched until final evaluation.**

## Class Imbalance and Augmentation

For highly imbalanced classification:

- Accuracy may be misleading.
- Use Precision, Recall, F1, PR-AUC or Balanced Accuracy.
- Oversample minorities or undersample majorities.
- Use class weights or focal loss.
- Tune the decision threshold when appropriate.

### Data augmentation

Images: crop, flip, rotate, color jitter, Cutout, MixUp.

Text/sequences: back-translation, controlled replacement, insertion/deletion.

**Augment training data only.**

## Dataset and DataLoader

| Component | Responsibility |
|---|---|
| `Dataset` | Returns individual samples and labels |
| `DataLoader` | Batching, shuffling, collation, parallel loading |

Useful DataLoader parameters:

- `batch_size`
- `num_workers`
- `pin_memory`
- `drop_last`

Common benchmark/data sources include CIFAR, ImageNet, COCO, UCI, Kaggle, Hugging Face datasets, GLUE, SQuAD and MovieLens.

---
# 3. Regression
---

## Linear Regression

Linear regression can be viewed as a **one-layer neural network without a nonlinear activation**.

## Fully Connected / Linear Layer

For one sample: $
\hat{\mathbf{y}}=\mathbf{W}^{\top}\mathbf{x}+\mathbf{b}
$

For a batch: $
\hat{\mathbf{Y}}=\mathbf{XW}+\mathbf{1}\mathbf{b}^{\top}$

Shapes:

$$
\mathbf{X}\in\mathbb{R}^{B\times D},\quad
\mathbf{W}\in\mathbb{R}^{D\times K},\quad
\mathbf{b}\in\mathbb{R}^{K},\quad
\hat{\mathbf{Y}}\in\mathbb{R}^{B\times K}.
$$

Libraries:

- PyTorch: `nn.Linear`
- Keras: `Dense`

## Closed-form Linear Regression

Include the bias using augmentation: $
\tilde{\mathbf{X}}=\begin{bmatrix}\mathbf{X}&\mathbf{1}\end{bmatrix},\qquad
\tilde{\mathbf{W}}=\begin{bmatrix}\mathbf{W}\\\mathbf{b}^{\top}\end{bmatrix}
$

Objective: $
\tilde{\mathbf{W}}^*=\arg\min_{\tilde{\mathbf{W}}}\|\tilde{\mathbf{X}}\tilde{\mathbf{W}}-\mathbf{Y}\|_F^2
$

Solution: $
\tilde{\mathbf{W}}=(\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^\top\mathbf{Y}
$

This requires matrix inversion and does not scale well to very high-dimensional models.

## Ridge Regression and Gradient Descent

![](https://miro.medium.com/1*nlvblFZKBOJVCyb7yDuQlQ.png)

Ridge regression adds L2 regularization:

$$
\tilde{\mathbf{W}}^*=\arg\min\left(\|\tilde{\mathbf{X}}\tilde{\mathbf{W}}-\mathbf{Y}\|_F^2+\lambda\|\mathbf{W}\|_F^2\right).
$$

Gradient descent instead updates iteratively:

$$
\mathbf{W}_{t+1}=\mathbf{W}_t-\eta\nabla_{\mathbf{W}}\mathcal{L},
$$

$$
\mathbf{b}_{t+1}=\mathbf{b}_t-\eta\nabla_{\mathbf{b}}\mathcal{L}.
$$

Gradient-based optimization generalizes naturally to deep neural networks.

---
# 4. Classification
---

Classification models produce **logits** and convert them into task-appropriate probabilities.

## Classification Architecture

Linear score:

$$
\mathbf{Z}=\mathbf{XW}+\mathbf{1}\mathbf{b}^{\top}.
$$

Probability output:

$$
\hat{\mathbf{Y}}=\operatorname{Activation}(\mathbf{Z}).
$$

| Task | Logits --- | Mapping |
|---|---|---|
| Binary | $(B,1)$ | Sigmoid |
| Single-label multiclass | $(B,K)$ | Softmax |
| Multi-label | $(B,K)$ | Sigmoid independently per label |

Same basic architecture, different learning problems.

## Example functions: Sigmoid and Softmax

### Sigmoid

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

Used for binary classification:

$$
\hat p=P(y=1\mid\mathbf{x}).
$$

### Softmax

$$
\hat p_k=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}.
$$

Properties resemble probability distribution:

$$
\hat p_k\ge0,\qquad \sum_{k=1}^{K}\hat p_k=1.
$$

Therefore softmax outputs a categorical probability distribution.

## Output Layer and Loss: Practical Convention

### PyTorch

Normally return **logits** from the network.

- `BCEWithLogitsLoss` → internally applies sigmoid.
- `CrossEntropyLoss` → internally handles softmax/log-softmax behavior.

Do not apply the activation twice.

### Keras

| Task | Last output | Loss |
|---|---|---|
| Binary | 1 sigmoid | Binary cross-entropy |
| Multiclass | $K$ softmax | Categorical cross-entropy |
| Multi-label | $K$ sigmoid | Binary cross-entropy |
| Regression | Linear | MSE |

---
# 5. Loss Functions & Entropy
---

A **loss function** defines what the optimizer tries to minimize.

A loss and an evaluation metric may use similar equations, but serve different purposes.

## Regression Losses

Residual: $
e_i=y_i-\hat y_i.
$

Mean Squared Error: $
\mathcal{L}_{MSE}=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2.
$

Mean Absolute Error: $
\mathcal{L}_{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|.
$

- MSE → quadratic penalty, emphasizes large errors.
- MAE → linear penalty, more robust to outliers.

## Label Representation

<img src="https://external-preview.redd.it/wait-thats-not-fried-chicken-v0-Hyah43KS-0sADMbQXze_LbhZLFd211X-qpyKfGhk0yQ.jpeg?auto=webp&s=5b721e0606d833d35d31ba195ae234730369da99" height=200 />

Class indices are symbols, not numerical magnitudes.

For $K$ classes, a one-hot label satisfies

$$
\mathbf{y}\in\{0,1\}^K,\qquad \sum_{k=1}^{K}y_k=1.
$$

Example:

$$
[0,1,0].
$$

A **soft label** may instead be

$$
[0.05,0.90,0.05].
$$

Sources include label smoothing, knowledge distillation and human ambiguity.

## Binary and Categorical Cross-Entropy

### Binary cross-entropy

$$
\mathcal{L}_{BCE}=-[y\log\hat p+(1-y)\log(1-\hat p)].
$$

### Categorical cross-entropy

$$
\mathcal{L}_{CE}=-\sum_{k=1}^{K}y_k\log \hat p_k.
$$

For one-hot true class $c$:

$$
\mathcal{L}_{CE}=-\log \hat p_c.
$$

Cross-entropy strongly penalizes confident incorrect predictions.

---
# 6. Evaluation Metrics
---

Loss functions guide training. Metrics quantify how useful the trained model is for the actual problem.

## Regression Metrics

| Metric | Formula --------- | Interpretation |
|---|---|---|
| MAE | $\frac1n\sum_i|y_i-\hat y_i|$ | Mean error magnitude |
| MSE | $\frac1n\sum_i(y_i-\hat y_i)^2$ | Strongly penalizes large errors |
| RMSE | $\sqrt{\mathrm{MSE}}$ | Same unit as target |
| $R^2$ | $1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}$ | Improvement over mean baseline |

For $R^2$:

- $1$ → perfect;
- $0$ → equal to predicting the mean;
- $<0$ → worse than the mean predictor.

## Binary Classification: Confusion Matrix

|  | Predicted Positive | Predicted Negative |
|---|---:|---:|
| Actual Positive | TP | FN |
| Actual Negative | FP | TN |

$$
\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}.
$$

$$
\text{Precision}=\frac{TP}{TP+FP},\qquad
\text{Recall}=\frac{TP}{TP+FN}.
$$

$$
F_1=2\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}.
$$

## Thresholds, Balanced Accuracy, Macro and Micro

Prediction threshold:

$$
\hat y=\mathbf{1}\{\hat p\ge\tau\}.
$$

Specificity:

$$
\text{Specificity}=\frac{TN}{TN+FP}.
$$

Balanced Accuracy:

$$
\text{BalancedAcc}=\frac12(\text{Recall}+\text{Specificity}).
$$

Macro-F1:

$$
\text{Macro-F1}=\frac{1}{K}\sum_{k=1}^{K}F1_k.
$$

Micro-F1:

$$
\text{Micro-F1}=\frac{2TP}{2TP+FP+FN}.
$$

- **Macro:** classes have equal importance.
- **Micro:** common classes have greater influence.

---
# 7. Model Capacity and Inductive Bias
---

A model can only learn functions contained in its hypothesis space.

## Capacity of Linear Models

Linear regression:

$$
\hat y=\mathbf{w}^{\top}\mathbf{x}+b.
$$

Logistic-regression boundary:

$$
\mathbf{w}^{\top}\mathbf{x}+b=0.
$$

Softmax class score:

$$
z_k=\mathbf{w}_k^{\top}\mathbf{x}+b_k.
$$

Boundary between classes $i,j$:

$$
(\mathbf{w}_i-\mathbf{w}_j)^\top\mathbf{x}+(b_i-b_j)=0.
$$

These models remain **linear in their feature space**.

## Underfitting vs Overfitting

### Underfitting

- model too simple;
- high training error;
- high validation error;
- high bias.

### Overfitting

- model too sensitive to training data;
- low training error;
- high validation error;
- high variance.

Conceptually,

$$
\mathbb{E}[(y-\hat f(x))^2]=\text{Bias}^2+\text{Variance}+\text{Noise}.
$$

The goal is **good generalization**, not maximum capacity.

## Feature Transformation and Representation Learning

![](https://graphics.cs.cmu.edu/projects/deepContext/images/teaser.jpg)

Define

$$
\Phi:\mathbb{R}^{D}\rightarrow\mathbb{R}^{M}.
$$

Then

$$
\hat y=\mathbf{w}^{\top}\Phi(\mathbf{x})+b.
$$

Deep networks learn $\Phi$ automatically:

$$
\Phi(\mathbf{x})=f_L(\cdots f_2(f_1(\mathbf{x}))).
$$

Examples:

- CNN → spatial features;
- RNN → temporal features;
- Transformer → contextual features;
- state-space models such as Mamba → sequence-state representations.

> **Deep learning = deep representation learning.**

---
# 8. Multilayer Perceptron (MLP)
---

<img src="https://cdn-images-1.medium.com/max/800/0*eaw1POHESc--l5yR.png" height=200 />

MLPs learn nonlinear feature transformations using fully connected layers and activation functions.

## MLP Architecture and Forward Pass

Conceptually:

$$
\mathbf{x}\rightarrow\text{Feature Transformer}\rightarrow\text{Output Head}\rightarrow\hat{\mathbf{y}}.
$$

Let

$$
\mathbf{h}^{(0)}=\mathbf{x}.
$$

Hidden layer $\ell$:

$$
\mathbf{h}^{(\ell)}=\phi\left(\mathbf{W}^{(\ell)}\mathbf{h}^{(\ell-1)}+\mathbf{b}^{(\ell)}\right).
$$

Without nonlinear activations, multiple FC layers collapse into a single FC layer.

## Output Heads and Parameter Count

| Task | Output head |
|---|---|
| Regression | Linear |
| Binary classification | One logit |
| Multiclass classification | $K$ logits |
| Multi-label classification | $K$ independent logits |

For

$$
\mathbf{x}\in\mathbb{R}^{N},\quad
\mathbf{W}\in\mathbb{R}^{M\times N},\quad
\mathbf{b}\in\mathbb{R}^{M},
$$

a fully connected layer contains

$$
\boxed{MN+M}
$$

trainable parameters.

## Activation Functions

| Activation | Typical role | Notes |
|---|---|---|
| Sigmoid | Binary output | Saturates |
| Tanh | Legacy hidden layers | Zero-centered but saturates |
| ReLU | Hidden layers | Simple; dead-neuron risk |
| Leaky ReLU | Hidden layers | Keeps negative gradient |
| SiLU / Swish | Modern hidden layers | Smooth |
| Softmax | Multiclass output | Produces distribution |

Rule of thumb:

- hidden layers → ReLU or variants;
- regression output → identity;
- classification output → determined by task/loss.

## BatchNorm and LayerNorm

BatchNorm:

$$
\hat x_i=\frac{x_i-\mu_B}{\sqrt{\sigma_B^2+\epsilon}},
$$

$$
y_i=\gamma\hat x_i+\beta.
$$

- **BatchNorm:** normalize using batch statistics.
- **LayerNorm:** normalize features within each sample.

Normalization can stabilize training and reduce sensitivity to initialization.

---
# 9. Training Neural Networks: Steps, Optimization
---

Training repeatedly modifies $\theta$ to reduce a loss. Optimization is used so that best result can be found sooner.

## Training Process

![](https://miro.medium.com/0*h7SARb2rUsVQGWwX.gif)

Goal:

$$
\theta^*=\arg\min_\theta\mathcal{L}(\theta).
$$

Three core steps:

1. **Forward:** compute prediction and loss.
2. **Backward:** compute gradients.
3. **Update:** modify parameters.

Pipeline:

$$
\text{Data}\rightarrow\text{Forward}\rightarrow\text{Loss}\rightarrow\text{Backward}\rightarrow\text{Update}\rightarrow\text{Repeat}.
$$

## Backpropagation and the Chain Rule

If a parameter $w$ affects the loss through several operations:

$$
\frac{\partial\mathcal{L}}{\partial w}=\frac{\partial\mathcal{L}}{\partial\hat y}\frac{\partial\hat y}{\partial z_1}\frac{\partial z_1}{\partial z_2}\cdots\frac{\partial z_k}{\partial w}.
$$

Each term is a local derivative.

Backpropagation efficiently reuses these derivatives through the computation graph.

Modern frameworks perform this through automatic differentiation.

## Training Terminology

| Term | Meaning |
|---|---|
| Mini-batch | Samples in one update |
| Iteration / step | One parameter update |
| Epoch | One pass through training data |
| Learning rate $\eta$ | Update step size |
| Gradient | Direction of steepest loss increase |
| Momentum | Running gradient direction |
| LR schedule | Changes $\eta$ during training |

Too large $\eta$ → unstable/divergent.

Too small $\eta$ → very slow training.

## Common Optimizers

<img src="https://cdn-images-1.medium.com/max/800/1*NRCWfdXa7b-ak2nBtmwRvw.png" height=200 />

| Optimizer | Idea | Typical use |
|---|---|---|
| SGD | Current gradient | Basic baseline |
| Momentum | Accumulated velocity | Vision / stable convergence |
| Nesterov | Momentum look-ahead | Reduce overshoot |
| AdaGrad | Accumulated squared gradients | Sparse problems |
| RMSProp | Moving squared-gradient average | RNN / nonstationary problems |
| Adam | Momentum + adaptive scaling | General-purpose |
| AdamW | Adam + decoupled weight decay | Modern default |

Adam tracks approximately

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,
$$

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.
$$

## Learning-Rate Scheduling

Cosine annealing:

$$
\eta_t=\frac{1}{2}\eta_0\left[1+\cos\left(\frac{t}{T}\pi\right)\right].
$$

Other common ideas:

- step decay;
- warmup;
- warm restarts;
- one-cycle schedules.

The learning-rate schedule can matter as much as optimizer choice.

## Regularization, Early Stopping and Initialization

L2: $
\mathcal{L}_{total}=\mathcal{L}+\frac{\lambda}{2}\|\theta\|_2^2
$

L1: $
\mathcal{L}_{total}=\mathcal{L}+\lambda\|\theta\|_1
$

Other techniques:

- Dropout
- BatchNorm / LayerNorm
- Early stopping
- Gradient clipping

Initialization:

| Activation | Common initialization |
|---|---|
| Sigmoid / Tanh | Xavier / Glorot |
| ReLU family | He / Kaiming |

## Batch Size Trade-off

### Small batches

- noisier gradients;
- lower memory;
- lower throughput;
- often good generalization.

### Large batches

- smoother gradients;
- better hardware throughput;
- higher memory use;
- may need learning-rate scaling and warmup.

Batch size is both a **hardware** and an **optimization** hyperparameter.

---
# Week 2 Summary
---

The complete pipeline is now

$$
\boxed{\text{Problem}\rightarrow\text{Dataset}\rightarrow\text{Representation}\rightarrow\text{Model}\rightarrow\text{Loss}\rightarrow\text{Optimization}\rightarrow\text{Evaluation}}
$$

Key connections:

- Problem formulation determines the task.
- Dataset preparation determines whether learning and evaluation are valid.
- Linear models provide the basic affine operation.
- Losses define the learning objective.
- Metrics define practical success.
- Model capacity explains under/overfitting.
- Feature transformation motivates representation learning.
- MLPs learn nonlinear representations.
- Backpropagation and optimizers make neural-network learning possible.

## Foundation Checklist

Before moving to CNNs, RNNs or Transformers, you should be able to answer:

1. What are $\mathbf{X}$, $\mathbf{Y}$, $\mathbf{W}$, $\mathbf{Z}$ and $\hat{\mathbf{Y}}$?
2. How do regression, binary, multiclass and multi-label tasks differ?
3. Why must preprocessing be fitted only on training data?
4. When should splitting be stratified or temporal?
5. What is the difference between a loss and a metric?
6. Why is logistic regression still a linear classifier?
7. Why are nonlinear activations required between FC layers?
8. How many parameters does an FC layer contain?
9. What happens during forward, backward and update?
10. How do optimizer, learning rate, regularization and batch size affect training?